# Tumor Stroma border
##### Franziska Niemeyer, 2025-10-21

In [ ]:
import pandas as pd
import os
import scanpy as sc
import anndata as ad
import numpy as np
import matplotlib.pyplot as plt
from aquarel import load_theme
import cmcrameri
cmap = cmcrameri.cm.lipari

In [ ]:
def mm_to_inches(mm):
    return mm / 25.4

def set_size(width="single", height_ratio=0.62):
    if width == "single":
        w_mm = 89
    elif width == "double":
        w_mm = 183
    elif isinstance(width, (int, float)):
        w_mm = float(width)
    else:
        raise ValueError("width must be 'single', 'double', or a number in mm")
    w_in = mm_to_inches(w_mm)
    h_in = w_in * height_ratio
    plt.gcf().set_size_inches(w_in, h_in)

cmap = cmcrameri.cm.lipari
color_cycle = [cmap(x) for x in (0.12, 0.28, 0.44, 0.60, 0.76, 0.90)]

nature_theme = (
    load_theme("umbra_light")
    .set_overrides({
        # Canvas
        "figure.facecolor": "white",
        "axes.facecolor": "white",

        # Spines (thin, only left/bottom)
        "axes.edgecolor": "black",
        "axes.linewidth": 0.6,
        "axes.spines.top": False,
        "axes.spines.right": False,

        # Ticks (outward, thin)
        "xtick.direction": "out",
        "ytick.direction": "out",
        "xtick.major.size": 3,
        "ytick.major.size": 3,
        "xtick.minor.size": 1.5,
        "ytick.minor.size": 1.5,
        "xtick.major.width": 0.6,
        "ytick.major.width": 0.6,
        "xtick.minor.width": 0.5,
        "ytick.minor.width": 0.5,
        "xtick.minor.visible": True,
        "ytick.minor.visible": True,

        # --- Grid (faint, only y-axis) ---
        "axes.grid": True,
        "axes.axisbelow": True,
        "grid.color": "#EAEAEA",
        "grid.linewidth": 0.5,
        "grid.linestyle": "-",
        "axes.grid.axis": "y",

        # Fonts (compact)
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "font.size": 15,          # base
        "axes.labelsize": 15,     # axis labels
        "axes.titlesize": 18,     # small, not oversized
        "xtick.labelsize": 15,
        "ytick.labelsize": 15,
        "legend.fontsize": 15,

        # Lines / markers / patches (slim)
        "lines.linewidth": 1,
        "lines.markersize": 10,
        "patch.linewidth": 0.6,
        "errorbar.capsize": 2,

        # Legend (no frame by default)
        "legend.frameon": False,
        "legend.borderaxespad": 0.8,

        # Color cycle
        "axes.prop_cycle": plt.cycler(color=color_cycle),

        # Save/export
        "savefig.dpi": 600,
        "savefig.bbox": "tight",
        "savefig.facecolor": "white",
        "savefig.edgecolor": "none",
    })
)

In [ ]:
OUT_DIR = "tumor-stroma_outputs"
INPUT_H5AD = "../../quality_control/primary-cohort/adata.h5ad"

if not os.path.exists(OUT_DIR):
    os.makedirs(OUT_DIR)
sc.settings.figdir = OUT_DIR
plt.rcParams['savefig.dpi'] = 600

#### Load and filter the data

In [ ]:
adata = ad.read_h5ad(INPUT_H5AD)

#### Preprocess the data
Grouping the data by class, patient, and PFI.
Summarize medium and long PFI as long.

In [ ]:
adata.obs['class'] = adata.obs['class'].astype(str)
adata.obs['class_overall'] = adata.obs['class'].replace({'OvaryR': 'Ovary', 'OvaryL': 'Ovary'})
adata.obs['PFI'] = adata.obs.PFI.astype(str)
adata.obs['PFI_short_long'] = adata.obs['PFI'].replace({'short': 'short', 'medium': 'long', 'long': 'long'})
adata.obs['patient'] = adata.obs['patient'].astype(str)
adata.obs['anno'] = adata.obs['class'] + '_' + adata.obs['patient'] + '_PFI-' + adata.obs['PFI']

In [ ]:
adata_backup = adata.copy()

In [ ]:
adata = adata[~adata.obs['class'].isin(['Marker'])].copy()

### Differential gene expression stats

In [ ]:
import numpy as np
import pandas as pd
import scikit_posthocs as sp
from scipy.stats import kruskal
from statsmodels.stats.multitest import multipletests

def _stars(p):
    if p is None or not np.isfinite(p): return "ns"
    return "***" if p < 1e-3 else "**" if p < 1e-2 else "*" if p < 5e-2 else "ns"

def kruskal_pfi_global(adata, genes, pfi_key="PFI", patient_key="patient",
                       pfi_order=("long", "medium", "short"), layer="log-transformed",
                       agg="median", pairwise_adjust="holm", omnibus_adjust="fdr_bh",
                       gate_posthoc=True):
    pfi_order = list(pfi_order)
    rows = []
    for gene in genes:
        vec = adata[:, gene].layers[layer] if layer else adata[:, gene].X
        vec = np.asarray(vec.toarray() if hasattr(vec, "toarray") else vec).ravel()
        df = pd.DataFrame({"expression": vec,
                           "pfi": adata.obs[pfi_key].values,
                           "patient": adata.obs[patient_key].values}).dropna()

        # one value per patient across all their spots
        pp = df.groupby(["patient", "pfi"], observed=True)["expression"].agg(agg).reset_index()
        groups = {s: pp.loc[pp["pfi"].astype(str) == s, "expression"].values for s in pfi_order}
        groups = {s: v for s, v in groups.items() if v.size}
        n_str = ", ".join(f"{s}={groups.get(s, []).__len__()}" for s in pfi_order)

        if len(groups) < 2:
            rows.append(dict(gene=gene, contrast="omnibus", n=n_str,
                             p_raw=np.nan, p_adj=np.nan, signif="ns")); continue

        H, p_kw = kruskal(*groups.values())
        rows.append(dict(gene=gene, contrast="omnibus", n=n_str, statistic=H,
                         p_raw=p_kw, p_adj=np.nan, signif=_stars(p_kw)))

        if (not gate_posthoc) or (np.isfinite(p_kw) and p_kw < 0.05):
            dunn = sp.posthoc_dunn(list(groups.values()), p_adjust=pairwise_adjust)
            labels = list(groups.keys())
            for i, a in enumerate(labels):
                for b in labels[i + 1:]:
                    padj = dunn.iloc[i, labels.index(b)]
                    rows.append(dict(gene=gene, contrast=f"{a} vs {b}",
                                     n=f"{a}={len(groups[a])}, {b}={len(groups[b])}",
                                     p_adj=padj, signif=_stars(padj)))

    out = pd.DataFrame(rows)
    m = (out["contrast"] == "omnibus") & out["p_raw"].notna()
    if m.any():
        out.loc[m, "p_adj"] = multipletests(out.loc[m, "p_raw"], method=omnibus_adjust)[1]
        out.loc[m, "signif"] = out.loc[m, "p_adj"].map(_stars)
    return out.sort_values(["gene", "contrast"]).reset_index(drop=True)

In [ ]:
pd.set_option("display.max_rows", None, "display.width", 200)

res = kruskal_pfi_global(
    adata,
    genes=["C3", "IFI27", "BST2"],
    pfi_key="PFI",
    patient_key="patient",
    pfi_order=["long", "medium", "short"],
    layer="log-transformed",
    agg="median",
)
print(res.to_string(index=False))

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import kruskal
from itertools import combinations
from statsmodels.stats.multitest import multipletests


def _p_to_stars(p):
    if p is None or not np.isfinite(p):
        return "ns"
    if p < 1e-3:
        return "***"
    if p < 1e-2:
        return "**"
    if p < 5e-2:
        return "*"
    return "ns"


def dunn_posthoc(groups, p_adjust="bonferroni"):
    """
    Dunn's test on a dict {label: 1d array of patient-level values}.
    Returns dict {(a, b): adjusted_p} for every pair.
    Rank-sums use the pooled ranking (correct Dunn's formulation),
    with tie correction. Pairwise p-values are corrected together.
    """
    labels = list(groups.keys())
    data = {k: np.asarray(v, float) for k, v in groups.items()}
    data = {k: v[np.isfinite(v)] for k, v in data.items()}
    ns = {k: v.size for k, v in data.items()}

    all_vals = np.concatenate([data[k] for k in labels])
    N = all_vals.size
    ranks = pd.Series(all_vals).rank().values  # average ranks -> ties handled

    # split ranks back per group
    rank_groups, idx = {}, 0
    for k in labels:
        rank_groups[k] = ranks[idx: idx + ns[k]]
        idx += ns[k]
    Rbar = {k: rank_groups[k].mean() for k in labels}

    # tie correction term
    _, counts = np.unique(all_vals, return_counts=True)
    ties = np.sum(counts**3 - counts)
    sigma2 = (N * (N + 1) - ties / (N - 1)) / 12.0

    from scipy.stats import norm
    pairs, raw_p = [], []
    for a, b in combinations(labels, 2):
        if ns[a] == 0 or ns[b] == 0:
            pairs.append((a, b)); raw_p.append(np.nan); continue
        se = np.sqrt(sigma2 * (1.0 / ns[a] + 1.0 / ns[b]))
        z = (Rbar[a] - Rbar[b]) / se
        raw_p.append(2 * norm.sf(abs(z)))
        pairs.append((a, b))

    raw_p = np.asarray(raw_p, float)
    ok = np.isfinite(raw_p)
    adj = np.full_like(raw_p, np.nan)
    if ok.any():
        adj[ok] = multipletests(raw_p[ok], method=p_adjust)[1]
    return {pair: adj[i] for i, pair in enumerate(pairs)}


def kruskal_with_dunn(df_means, group_col, subset_col, value_col="expression",
                      subset_order=None, p_adjust="holm"):
    """
    For each level of `group_col`, run Kruskal-Wallis across `subset_col`
    on patient-level means, then Dunn's post-hoc.
    """
    out = {}
    for g, gdf in df_means.groupby(group_col, observed=True):
        subs = (subset_order if subset_order is not None
                else list(gdf[subset_col].dropna().unique()))
        groups = {}
        for s in subs:
            v = gdf.loc[gdf[subset_col].astype(str) == str(s), value_col].dropna().values
            if v.size > 0:
                groups[s] = v
        n = {s: groups.get(s, np.array([])).size for s in subs}

        if len(groups) < 2 or any(v.size < 1 for v in groups.values()):
            out[g] = {"H": np.nan, "p_kw": np.nan, "n": n, "pairwise": {}}
            continue
        try:
            H, p_kw = kruskal(*groups.values())
        except ValueError:
            H, p_kw = np.nan, np.nan

        # only run post-hoc if omnibus is significant
        pairwise = (dunn_posthoc(groups, p_adjust=p_adjust)
                    if np.isfinite(p_kw) and p_kw < 0.05 else {})
        out[g] = {"H": H, "p_kw": p_kw, "n": n, "pairwise": pairwise}
    return out

In [ ]:
def _add_pairwise_brackets(ax, positions, pvals, y0, *, heights=None,
                           text_fn=_p_to_stars, lw=1, drop="ns",
                           pad_frac=0.03, step_frac=0.13, fontsize=13,
                           max_leg_frac=0.9):
    """
    Draws one bracket per significant pair, widest span highest.
    """
    yr0, yr1 = ax.get_ylim()
    yrng = yr1 - yr0
    pad = pad_frac * yrng
    step = step_frac * yrng
    max_leg = max_leg_frac * yrng

    items = []
    for (a, b), p in pvals.items():
        if a not in positions or b not in positions:
            continue
        s = text_fn(p)
        if drop and s == drop:
            continue
        xa, xb = positions[a], positions[b]
        left_lab, right_lab = (a, b) if xa <= xb else (b, a)
        items.append((abs(xb - xa), min(xa, xb), max(xa, xb),
                      left_lab, right_lab, s))
    # draw narrowest first (lowest), widest last (highest) to avoid overlap
    items.sort(key=lambda t: t[0])

    level = 0
    for _, xl, xr, left_lab, right_lab, s in items:
        yh = y0 + pad + level * step + pad * 0.5   # horizontal line height
        y_tick = yh - pad * 0.5                     # fixed short leg bottom

        def _leg_bottom(h, other_h):
            # taller bar (or missing heights) keeps the fixed short tick;
            # only the shorter bar's leg drops down toward it
            if heights is None or not np.isfinite(h) or not np.isfinite(other_h):
                return y_tick
            if h >= other_h:
                return y_tick
            return min(y_tick, max(yh - max_leg, h + pad * 0.4))

        hl = heights.get(left_lab, np.nan) if heights is not None else np.nan
        hr = heights.get(right_lab, np.nan) if heights is not None else np.nan
        yl_bot = _leg_bottom(hl, hr)
        yr_bot = _leg_bottom(hr, hl)

        ax.plot([xl, xl, xr, xr], [yl_bot, yh, yh, yr_bot],
                c="k", lw=lw, zorder=6, clip_on=False)
        ax.text((xl + xr) / 2.0, yh, s, ha="center", va="bottom",
                color="k", fontsize=fontsize, zorder=7, fontweight='bold', clip_on=False)
        level += 1
    return y0 + pad + max(level, 1) * step

In [ ]:
def plot_gene_patient_boxplots(
    adata,
    gene_name,
    groupby,
    layer='log-transformed',
    agg="median",
    subset_by=None,
    group_order = ["Stroma", "Mixed", "Tumor Epi"],
    subset_order=None,
    patient_key=None,
    *,
    annotate_sig=True,
    skip_groups=None,
    sig_ref_index=2,
    sig_text="***",
    sig_pad=0.02,
    sig_lw=1,
    palette=None,
    cmap=cmcrameri.cm.lipari,
    theme=None,
    figsize=(5, 5),
    ax=None,
    title=None,
    show=True,
    dot_size=20,
    dot_alpha=0.9,
    bar_alpha = 0.3,
    rng=None,
    dot_shape_by=None,
    dot_markers=None
):
    if groupby not in adata.obs.columns:
        raise KeyError(f"`groupby='{groupby}'` not found in adata.obs.")
    if subset_by is not None and subset_by not in adata.obs.columns:
        raise KeyError(f"`subset_by='{subset_by}'` not found in adata.obs.")

    if gene_name not in adata.var_names:
        raise KeyError(f"Gene '{gene_name}' not in `adata.var_names`.")
    gene_vec = adata[:, gene_name].layers[layer]

    if hasattr(gene_vec, "toarray"):
        gene_vec = gene_vec.toarray()
    gene_vec = np.asarray(gene_vec).ravel()

    if patient_key is None:
        for cand in ("patient_id", "donor", "patient", "sample_id", "subject_id"):
            if cand in adata.obs.columns:
                patient_key = cand
                break
        else:
            patient_key = "_tmp_row_id"

    df = pd.DataFrame({
        "expression": gene_vec,
        "group": adata.obs[groupby].values
    }, index=adata.obs_names)

    if patient_key == "_tmp_row_id":
        df[patient_key] = np.arange(adata.n_obs)
    else:
        df[patient_key] = adata.obs[patient_key].values

    if subset_by is not None:
        df[subset_by] = adata.obs[subset_by].values

    if dot_shape_by is not None:
        if dot_shape_by not in adata.obs.columns:
            raise KeyError(f"`dot_shape_by='{dot_shape_by}'` not found in adata.obs.")
        df["_tissue"] = adata.obs[dot_shape_by].values

    gb_cols = ["group", patient_key]
    if subset_by is not None:
        gb_cols.append(subset_by)
    if dot_shape_by is not None:
        gb_cols.append("_tissue")

    means = (
        df.dropna(subset=["group"])
          .groupby(gb_cols, observed=True)["expression"]
          .agg(agg)
          .reset_index()
    )
    if groupby == "histology":
        means["group"] = pd.Categorical(means["group"], categories=group_order, ordered=True)

    col_expr = "expression"
    if dot_shape_by is not None:
        means["_tissue"] = means["_tissue"].astype(str)

    overall_means = means.groupby("group", observed=True)[col_expr].agg(agg).reindex(group_order)
    order = overall_means.index.tolist()
    x = np.arange(len(order))

    if subset_by is not None:
        if subset_order is not None:
            subset_levels = [str(s) for s in subset_order
                             if str(s) in means[subset_by].astype(str).unique()]
        else:
            subset_levels = means[subset_by].dropna().astype(str).unique().tolist()
    else:
        subset_levels = ["All"]

    if palette is not None:
        if subset_by is not None:
            # map per subgroup level
            thin_colors = [palette.get(s, "#cccccc") for s in subset_levels]
        else:
            # map per main group
            thin_colors = [palette.get(g, "#cccccc") for g in order]
    else:
        # fallback to cmap-based coloring
        if cmap is None:
            cmap = plt.cm.viridis
        thin_colors = [cmap(i / (len(subset_levels) + 1)) for i in range(len(subset_levels))]

    if dot_shape_by is not None:
        tissue_levels = means["_tissue"].dropna().astype(str).unique().tolist()
        if dot_markers is None:
            default_cycle = ['o', 's', '^', 'D', 'P', 'X', 'v', '<', '>', 'H', '*']
            dot_markers = {c: default_cycle[i % len(default_cycle)] for i, c in enumerate(tissue_levels)}
        else:
            default_cycle = ['o', 's', '^', 'D', 'P', 'X', 'v', '<', '>', 'H', '*']
            for i, c in enumerate(tissue_levels):
                dot_markers.setdefault(c, default_cycle[i % len(default_cycle)])
    else:
        tissue_levels = []

    if rng is None or isinstance(rng, (int, np.integer)):
        rng = np.random.default_rng(rng)

    ctx = theme if theme is not None else nullcontext()
    with ctx:
        created_fig = False
        if ax is None:
            fig, ax = plt.subplots(figsize=figsize)
            created_fig = True
        else:
            fig = ax.figure

        main_width = 1
        if subset_by is not None:
            box_width = main_width / (len(subset_levels) + 1)
            start_offset = -main_width / 2 + box_width / 2
        else:
            box_width = main_width * 0.6
            start_offset = 0.0

        show_background_bars = True
        if show_background_bars:
            if subset_by is not None:
                subset_means = (means.groupby(["group", subset_by], observed=True)[col_expr]
                                .agg(agg).unstack(subset_by))
                subset_means = subset_means.reindex(order)
            else:
                subset_means = means.groupby("group", observed=True)[col_expr].agg(agg)

            for gi, g in enumerate(order):
                if subset_by is not None:
                    for si, s in enumerate(subset_levels):
                        if (g in subset_means.index) and (s in subset_means.columns):
                            mean_val = subset_means.loc[g, s]
                            if pd.notna(mean_val):
                                offset = x[gi] + start_offset + si * box_width
                                left  = offset - box_width / 2
                                right = offset + box_width / 2
                                # ax.hlines(mean_val, left, right, lw=1, ls='--', color='k', alpha=0.7, label=None)
                                ax.bar(offset, mean_val, width=box_width * 0.9,
                                       color=thin_colors[si], alpha=bar_alpha, zorder=2)
                else:
                    mean_val = subset_means.loc[g]
                    left  = x[gi] - (box_width * 0.5)
                    right = x[gi] + (box_width * 0.5)
                    ax.hlines(mean_val, left, right, lw=1, ls='--', color='k', alpha=0.7, label=None)
                    ax.bar(x[gi], mean_val, width=box_width, color=thin_colors[0],
                           alpha=bar_alpha, zorder=2)

        for gi, g in enumerate(order):
            gdf = means[means["group"] == g]
            for si, s in enumerate(subset_levels):
                if subset_by is None:
                    sdf = gdf.copy()
                else:
                    sdf = gdf[gdf[subset_by].astype(str) == str(s)]

                vals = sdf[col_expr].dropna().values
                if len(vals) == 0:
                    continue
                pos = x[gi] + (start_offset + si * box_width if subset_by is not None else 0.0)

                if dot_shape_by is None:
                    jitter = rng.normal(0.0, box_width/6, size=len(vals))
                    ax.scatter(np.full(len(vals), pos) + jitter, vals,
                               s=dot_size, alpha=dot_alpha, edgecolors="none",
                               color=thin_colors[si], zorder=4,
                               label=(f"{subset_by} = {s}" if subset_by is not None and gi == 0 else None))
                else:
                    for cls in tissue_levels:
                        sub = sdf[sdf["_tissue"] == cls]
                        if sub.empty:
                            continue
                        v = sub[col_expr].values
                        jitter = rng.normal(0.0, box_width/6, size=len(v))
                        ax.scatter(np.full(len(v), pos) + jitter, v,
                                   s=dot_size, alpha=dot_alpha, edgecolors="none",
                                   color=thin_colors[si], marker=dot_markers[cls], zorder=5,
                                   label=(f"{subset_by} = {s}" if subset_by is not None and gi == 0 else None))

        # --- Significance: Kruskal-Wallis + Dunn per histology group ---
        if annotate_sig and subset_by is not None:
            stats = kruskal_with_dunn(
                means, group_col="group", subset_col=subset_by,
                value_col=col_expr, subset_order=subset_levels, p_adjust="bonferroni",
            )

            def subgroup_xpos(gi, si):
                return x[gi] + (start_offset + si * box_width)

            for gi, g in enumerate(order):
                if skip_groups is not None:
                    if isinstance(skip_groups[0], (int, np.integer)):
                        if gi in skip_groups:
                            continue
                    elif g in skip_groups:
                        continue
                res = stats.get(g)
                if not res or not res["pairwise"]:
                    continue
                positions = {s: subgroup_xpos(gi, si)
                             for si, s in enumerate(subset_levels)}
                # baseline: top of the tallest bar/dot in this group
                gdf = means[means["group"] == g]
                y0 = gdf[col_expr].max()
                sub_tops = {
                    s: gdf.loc[gdf[subset_by].astype(str) == str(s), col_expr].max()
                    for s in subset_levels
                }
                _add_pairwise_brackets(ax, positions, res["pairwise"], y0, max_leg_frac=.9,
                                       heights=sub_tops, lw=sig_lw, fontsize=15)


        if title is None:
            title = f"{gene_name}"
        ax.set_title(title)
        # ax.set_xlabel(groupby)
        ax.set_ylabel(f"{agg.capitalize()} normalized expression")
        ax.set_xticks(x); ax.set_xticklabels(order, rotation=45, ha="right")
        
    return fig, ax, means

In [ ]:
rename_histology = {
    "Mixed TumEpi+Other": "Mixed",
    "Other": "Stroma",
    "Tumor Epithelium": "Tumor Epi"
}

adata.obs["histology"] = adata_backup.obs["histology"].cat.rename_categories(rename_histology)
adata.obs["histology"].value_counts()

In [ ]:
from matplotlib.ticker import FormatStrFormatter, MaxNLocator
from matplotlib.lines import Line2D
import matplotlib.patches as mpatches

genes = ["C3", "IFI27", "BST2"]

for g in genes:
    fig, ax, df_means = plot_gene_patient_boxplots(
        adata,
        layer='log-transformed',
        gene_name=g,
        agg="mean",
        figsize=(6,4),
        groupby="histology",
        group_order=["Stroma", "Mixed", "Tumor Epi"],
        subset_by="PFI",
        subset_order=["long", "medium", "short"],
        patient_key="patient",
        palette={'long': '#51AFA9', 'medium': '#1A71B8', 'short': '#DF7B26'},
        theme=nature_theme,
        dot_shape_by="class",
        dot_markers={"Ovary": "o", "Omentum": "^"},
        rng=42,
    )
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_major_formatter(FormatStrFormatter('%.1f'))
    fig.savefig(f"figures/PFI-groups_{g}.pdf", bbox_inches="tight")
    fig.savefig(f"figures/PFI-groups_{g}.png", dpi =600, bbox_inches="tight")
    plt.show()

    palette = {'long': '#51AFA9', 'medium': '#1A71B8', 'short': '#DF7B26'}
    dot_markers = {"Ovary": "o", "Omentum": "^"}
    
    # color = PFI group
    pfi_handles = [
        Line2D([0], [0], marker='o', linestyle='none', markersize=7,
               markerfacecolor=c, markeredgecolor='none', label=label.title())
        for label, c in palette.items()
    ]
    
    # marker shape = tissue class
    tissue_display = {"Ovary": "Adnexa", "Omentum": "Omentum"}
    
    shape_handles = [
        Line2D([0], [0], marker=m, linestyle='none', markersize=7,
               markerfacecolor='0.4', markeredgecolor='none',
               label=tissue_display.get(cls, cls).title())
        for cls, m in dot_markers.items()
    ]
        
    fig_leg, ax_leg = plt.subplots(figsize=(2, 2.5))
    ax_leg.axis('off')
    
    leg1 = ax_leg.legend(
        handles=pfi_handles, title='PFI',
        loc='upper left', bbox_to_anchor=(0, 1),
        frameon=False, fontsize=11, title_fontsize=12,
    )
    ax_leg.add_artist(leg1)
    
    ax_leg.legend(
        handles=shape_handles, title='Site',
        loc='upper left', bbox_to_anchor=(0, 0.5),
        frameon=False, fontsize=11, title_fontsize=12,
    )
    
    fig_leg.savefig('figures/PFI-groups_legend.pdf', bbox_inches='tight')
    fig_leg.savefig('figures/PFI-groups_legend.png', dpi=600, bbox_inches='tight')
    plt.show()
    
    plt.close(fig)

In [ ]:
def stats_table(df_means, group_col="group", subset_col=None, value_col="expression",
                subset_order=None, p_adjust="holm", gene_name=None,
                run_posthoc_regardless=False):
    """
    Full stats table for one gene: Kruskal-Wallis per group + Dunn's pairwise.
    """
    rows = []
    res = kruskal_with_dunn(
        df_means, group_col=group_col, subset_col=subset_col,
        value_col=value_col, subset_order=subset_order, p_adjust=p_adjust,
    )

    for g, r in res.items():
        n = r["n"]
        n_str = ", ".join(f"{k}={v}" for k, v in n.items())

        # omnibus row
        rows.append({
            "gene": gene_name,
            "group": g,
            "test": "Kruskal-Wallis",
            "contrast": "omnibus",
            "n": n_str,
            "statistic": r["H"],
            "p_raw": r["p_kw"],
            "p_adj": np.nan,
            "signif": _p_to_stars(r["p_kw"]),
        })

        # pairwise rows
        pairwise = r["pairwise"]
        if not pairwise and run_posthoc_regardless and np.isfinite(r["p_kw"]):
            # force post-hoc even if omnibus was n.s. (off by default)
            subs = subset_order or list(n.keys())
            groups = {}
            for s in subs:
                v = df_means.loc[
                    df_means[subset_col].astype(str) == str(s), value_col
                ].dropna().values if subset_col else np.array([])
                if v.size:
                    groups[s] = v
            pairwise = dunn_posthoc(groups, p_adjust=p_adjust) if len(groups) >= 2 else {}

        for (a, b), padj in pairwise.items():
            rows.append({
                "gene": gene_name,
                "group": g,
                "test": f"Dunn ({p_adjust})",
                "contrast": f"{a} vs {b}",
                "n": f"{a}={n.get(a, 0)}, {b}={n.get(b, 0)}",
                "statistic": np.nan,
                "p_raw": np.nan,       # dunn_posthoc returns only adjusted p
                "p_adj": padj,
                "signif": _p_to_stars(padj),
            })

    df = pd.DataFrame(rows)
    if not df.empty:
        df = df.sort_values(["gene", "group", "contrast"]).reset_index(drop=True)
    return df

In [ ]:
pd.set_option("display.max_rows", None, "display.width", 200)

genes = ["C3", "IFI27", "BST2"]
all_tables = []
for g in genes:
    fig, ax, df_means = plot_gene_patient_boxplots(
        adata, gene_name=g, groupby="histology", agg="mean",
        group_order=["Stroma", "Mixed", "Tumor Epi"],
        subset_by="PFI", subset_order=["long", "medium", "short"],
        patient_key="patient", palette={'long': '#51AFA9', 'medium': '#1A71B8', 'short': '#DF7B26'},
        theme=nature_theme, dot_shape_by="class",
        dot_markers={"Ovary": "o", "Omentum": "^"}, rng=42, show=False,
    )
    plt.close(fig)
    tbl = stats_table(
        df_means, group_col="group", subset_col="PFI",
        subset_order=["long", "medium", "short"], p_adjust="holm", gene_name=g,
    )
    all_tables.append(tbl)

full = pd.concat(all_tables, ignore_index=True)
full